In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
!pip install langchain_community

In [3]:
!pip install pypdf

In [4]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

/tmp/ipykernel_6515/265351730.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [5]:
print(pages[3].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
3. Understanding Data Plans and Fair Use Policy
Data plans define how much high-speed data a customer can consume per billing cycle. Understanding plan
mechanics helps agents resolve billing disputes and set correct customer expectations.
High-Speed Data Allowance: Each plan includes a fixed high-speed data allowance (e.g. 20 GB, 50 GB, or
Unlimited). Once this allowance is consumed, the account is throttled to a reduced speed  - typically 5


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
)

chunks = splitter.split_documents(pages)
len(chunks)

37

In [7]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [8]:
chunks[1].page_content

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app'

In [9]:
!pip install langchain_huggingface langchain_chroma

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks,embeddings)

print(f"Vector Store ready. {vector_store._collection.count()} vectors stored.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector Store ready. 37 vectors stored.


In [20]:
retriever = vector_store.as_retriever(search_kwargs = {"k": 3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i,doc in enumerate(retrieved):
  print(f"--- Chunk {i+1}")

  print(f"{i+1}. {doc.page_content[:200]} \n")

--- Chunk 1
1. Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace t 

--- Chunk 2
2. voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings 

--- Chunk 3
3. prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, c 



In [23]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00


In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

In [21]:
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""


In [25]:
def format_docs(docs):
  return "\n\n".join(d.page_content for d in docs)

In [26]:
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format = "parsed",
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Rag chain assembled")

Rag chain assembled


In [27]:
question = "How does internation roaming work adn what charges should i expect?"
print(f"Q: {question}\n")
print("A: ", chain.invoke(question))

Q: How does internation roaming work adn what charges should i expect?

A:  International roaming allows your device to connect to partner networks in foreign countries when outside your home network's coverage. Here's how it works and what charges to expect:

### **How It Works**  
1. **Authentication**: The visited country’s network verifies your subscription via signaling protocols (SS7/Diameter) with your home network.  
2. **Service Authorization**: Your home network validates your account and authorizes usage.  
3. **Billing**: All data, voice, and SMS traffic are sent back to your home network for billing, which may introduce latency compared to local networks.  

### **Charges**  
- **Roaming Zones**:  
  - **Zone A** (EU, UK, Australia, New Zealand): Lowest rates.  
  - **Zone B** (USA, Canada, Japan, Singapore): Moderate rates.  
  - **Zone C** (Rest of the world): Highest per-MB and per-minute charges.  
- **Roaming Bundles**: Always purchase a pre-paid roaming bundle before